# Fluids Examples

This is a jupyter notebook. The "real" notebook can be found [here](https://github.com/polyfem/polyfem.github.io/blob/docs/docs/fluids.ipynb).

Some imports for plotting

In [ ]:
import meshplot as mp
import numpy as np
import polyfempy as pf
import utils

## Stokes

Create the mesh using the utility function

In [ ]:
pts, faces = utils.create_quad_mesh(50)

create settings, pick linear $Q_2$ elements for velocity and $Q_1$ for pressure and select stokes as material model.

In [ ]:
settings = pf.Settings(
    discr_order=2,
    pressure_discr_order=1,
    
    pde=pf.PDEs.Stokes
)

Set the viscosity of the fluid

In [ ]:
settings.set_material_params("viscosity", 1)

The default solver do not support mixed formulation, we need to choose `UmfPackLU`

In [ ]:
settings.set_advanced_option("solver_type", "Eigen::UmfPackLU")

We use the standard [Driven Cavity](https://www.cfd-online.com/Wiki/Lid-driven_cavity_problem) problem

In [ ]:
problem = pf.DrivenCavity()

we set the problem

In [ ]:
settings.problem = problem

We create the solver and set the settings

In [ ]:
solver = pf.Solver()
solver.settings(settings)

This time we are using pts and faces instead of loading from the disk

In [ ]:
solver.set_mesh(pts, faces, vismesh_rel_area=0.001)

Solve!

In [ ]:
solver.solve()

We now get the solution and the pressure

In [ ]:
pts, tris, velocity = solver.get_sampled_solution()

and plot it!

In [ ]:
each = 10
p = mp.plot(pts, tris, np.linalg.norm(velocity, axis=1))
p.add_lines(pts[0:pts.shape[0]:each,:], pts[0:pts.shape[0]:each,:]+velocity[0:pts.shape[0]:each,:]/6)
p

## Navier-Stokes

Setup problem

In [ ]:
settings = pf.Settings(
    discr_order=2,
    pressure_discr_order=1,
    
    BDF_order=3,
    
    tend=8,
    time_steps=10,
    
    pde=pf.PDEs.NavierStokes
)

settings.set_material_params("viscosity", 0.001)
settings.set_advanced_option("solver_type", "Eigen::UmfPackLU")

problem = pf.FlowWithObstacle(U=1.5, time_dependent=True)
settings.problem = problem

Solve

In [ ]:
solver = pf.Solver()
solver.settings(settings)

solver.load_mesh_from_path("ns.obj", vismesh_rel_area=0.001)
solver.solve()

Plot

In [ ]:
pts = solver.get_sampled_points_frames()
tris = solver.get_sampled_connectivity_frames()
velocity = solver.get_sampled_solution_frames()

In [ ]:
each = 1
def plot(frame, p=None):
    pp = pts[frame]
    tt = tris[frame]
    vv = velocity[frame]
    p1 = mp.plot(pp, tt, np.linalg.norm(vv, axis=1), plot=p)
    if p is None:
        p=p1
    p.add_lines(pp[0:pp.shape[0]:each,:], pp[0:pp.shape[0]:each,:]+vv[0:pp.shape[0]:each,:]/10)
    return p

In [ ]:
plot(8)

In [ ]:
p = plot(0)

@mp.interact(frame=(0, len(velocity)-1))
def step(frame=0):
    plot(frame, p)